# Camada Bronze — Ingestão dos Dados do Olist

Neste notebook vamos ingerir os dados brutos do e-commerce Olist na camada Bronze.

São **9 arquivos CSV** (vindos do Kaggle, armazenados no volume landing) + **1 ingestão via API** (cotação do dólar do Banco Central).

Os dados são salvos exatamente como vieram da fonte, sem nenhuma transformação de negócio. A única coluna adicionada é `timestamp_ingestion`, que registra o momento exato da ingestão.

In [0]:
catalogo = "medalhao"
bronze_db = "bronze"
landing_path = f"/Volumes/{catalogo}/default/landing"

### Função de ingestão genérica
Lê o CSV do volume landing, adiciona o `timestamp_ingestion` e salva como tabela Delta na camada Bronze.

In [0]:
from pyspark.sql import functions as F

def ingest_csv(nome_arquivo, nome_tabela):
    try:
        filepath = f"{landing_path}/{nome_arquivo}"

        df = spark.read.csv(filepath, header=True, inferSchema=True)

        if df.count() == 0:
            raise ValueError(f"Arquivo {nome_arquivo} está vazio.")

        df = df.withColumn("timestamp_ingestion", F.current_timestamp())

        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"{catalogo}.{bronze_db}.{nome_tabela}")

        print(f"✅ {nome_tabela} criada com sucesso ({df.count()} registros)")

    except Exception as e:
        print(f"❌ Erro ao processar {nome_tabela}: {str(e)}")

### Ingestão dos 9 CSVs do Olist
Mapeamento conforme o enunciado da atividade: arquivo original → nome da tabela Bronze.

In [0]:
tabelas_csv = [
    ("olist_customers_dataset.csv",                "tb_customers"),
    ("olist_geolocation_dataset.csv",              "tb_geolocalizacao"),
    ("olist_order_items_dataset.csv",              "tb_order_items"),
    ("olist_order_payments_dataset.csv",           "tb_order_payments"),
    ("olist_order_reviews_dataset.csv",            "tb_order_reviews"),
    ("olist_orders_dataset.csv",                   "tb_orders"),
    ("olist_products_dataset.csv",                 "tb_products"),
    ("olist_sellers_dataset.csv",                  "tb_sellers"),
    ("product_category_name_translation.csv",      "tb_product_category_name_translation"),
]

for arquivo, tabela in tabelas_csv:
    ingest_csv(arquivo, tabela)

print("\nIngestão dos CSVs na camada Bronze concluída!")

### Ingestão via API — Cotação do Dólar (Banco Central)

Aqui extraímos a cotação de compra do dólar pela API PTAX do Banco Central do Brasil.

As datas de início e fim são passadas como parâmetros do notebook (widgets), no formato `MM-DD-AAAA` conforme exigido pela API.

In [0]:
dbutils.widgets.text("data_inicio", "01-01-2016", "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "12-31-2018", "Data Fim (MM-DD-AAAA)")

In [0]:
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

url = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    f"&$select=dataHoraCotacao,cotacaoCompra"
    f"&$format=json"
)

print(f"Consultando API: {data_inicio} até {data_fim}")

response = requests.get(url)
response.raise_for_status()
dados = response.json()["value"]

print(f"Registros retornados pela API: {len(dados)}")

schema = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True)
])

df_dolar = spark.createDataFrame(dados, schema)

df_dolar = df_dolar.withColumn("timestamp_ingestion", F.current_timestamp())

df_dolar.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{bronze_db}.tb_cotacao_dolar")

print(f"✅ tb_cotacao_dolar criada com sucesso ({df_dolar.count()} registros)")

### Validação — Conferindo as tabelas criadas na camada Bronze

In [0]:
%sql

SHOW TABLES IN medalhao.bronze;

In [0]:
tabelas_bronze = [
    "tb_customers", "tb_geolocalizacao", "tb_order_items",
    "tb_order_payments", "tb_order_reviews", "tb_orders",
    "tb_products", "tb_sellers", "tb_product_category_name_translation",
    "tb_cotacao_dolar"
]

for tabela in tabelas_bronze:
    qtd = spark.table(f"{catalogo}.{bronze_db}.{tabela}").count()
    print(f"{tabela}: {qtd} registros")